In [1]:
# filepath: notebooks/task2v3.ipynb
from pathlib import Path
import sys

def find_project_root():
    here = Path.cwd()
    for p in [here, *here.parents]:
        if (p / "utils").exists() and (p / "models").exists():
            return p
    for p in [here, *here.parents]:
        if any((p / m).exists() for m in (".git", "requirements.txt", "pyproject.toml")):
            return p
    return here

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import plotly.graph_objects as go
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from torch.amp import autocast, GradScaler

from utils.reproducibility_utils import seed_everything
from utils.data_loading import load_data, load_test_data
from utils.ImbalancedSampling import MixupAugmentation, TemporalAugmentation
from models.lstmv2 import LSTMModel
from utils.LDS_FocalR_Loss import EnhancedLDS_FocalR_Loss

Project root: e:\Automation_IT\3\Drift case study\MAIT_G2_Drift_Challenge


In [2]:

CONFIG = {
    # PATHS
    'TRAIN_PATH': "../datasets_challenge/train",
    'TEST_PATH': "../datasets_challenge/test",
    'DIST_CSV': "../datasets_challenge/sideSlip_distribution.csv",
    'CHECKPOINT_NAME': "best_model.pth",
    'SUBMISSION_CSV': "predictions.csv",
    
    # REPRODUCIBILITY
    'SEED': 42,
    
    # DATA LOADING
    'TRAIN_RATIO': 0.8,
    'BATCH_SIZE': 512,
    'SEQUENCE_LENGTH': 20,
    'THRESHOLD': True,
    'THRESHOLD_VALUE': 3.0,
    'THRESHOLD_COLUMN': "vxCG",
    'COLUMNS_TO_DROP': ["sideSlip", "vyCG", "time", "longitude", "latitude"],
    'TARGET': "sideSlip",
    'SCALING': True,
    
    # MODEL ARCHITECTURE 
    'MODEL_TYPE': 'BiLSTM_Attention',
    'RNN_UNITS': 128,
    'FC_UNITS': 128,
    'NUM_LAYERS': 2,
    'DROPOUT': 0.3,
    'BIDIRECTIONAL': True,
    'USE_ATTENTION': True,
    'FEATURE_DIM': 128,
    
    # LOSS FUNCTION 
    'BUCKET_WEIGHTS': (1.0, 3.0, 8.0),  
    'LDS_KERNEL': 'gaussian',
    'LDS_KS': 7,
    'LDS_SIGMA': 2.0,
    'FOCAL_GAMMA': 1.0,
    'USE_HUBER': True,
    'HUBER_BETA': 0.08,
    'NUM_BINS': 100,
    
    # AUGMENTATION 
    'MIXUP_ALPHA': 0.2,
    'MIXUP_PROB': 0.2,                   
    'MIXUP_PRIORITIZE_RARE': False,       
    'MIXUP_RARE_THRESHOLD': 5.0,
    'TEMPORAL_NOISE_PROB': 0.3,        
    'TEMPORAL_NOISE_STD': 0.01,          
    'TEMPORAL_MASK_PROB': 0.15,           
    'TEMPORAL_MASK_RATIO': 0.1,           
    'TEMPORAL_SCALE_PROB': 0.2,           
    'TEMPORAL_SCALE_RANGE': (0.95, 1.05), 
    'AUGMENTATION_EPOCHS': 80,           
    
    # OPTIMIZER - 
    'LEARNING_RATE': 1.5e-3,            
    'WEIGHT_DECAY': 1e-3,
    'BETAS': (0.9, 0.999),
    
    # SCHEDULER
    'SCHEDULER_T_MAX': 200,
    'SCHEDULER_ETA_MIN': 1e-6,
    
    # TRAINING 
    'NUM_EPOCHS': 200,
    'EARLY_STOP_PATIENCE': 25,           
    'GRADIENT_CLIP': 1.0,
    'USE_AMP': True,
    
    # CHALLENGE CONSTRAINTS
    'MANY_SHOT_CONSTRAINT': 0.2,
    'FEW_SHOT_TARGET': 0.3,
    
    # TEST-TIME AUGMENTATION 
    'TTA_N_AUGMENTATIONS': 3,            
    'TTA_NOISE_LEVELS': [0.005, 0.01, 0.015],  
    'TTA_ORIGINAL_WEIGHT': 2.0,           
    
    # VISUALIZATION
    'FIG_SIZE': (14, 8),
    'PLOTLY_WIDTH': 1200,
    'PLOTLY_HEIGHT': 500,
}


In [3]:
# filepath: notebooks/task2v3.ipynb
seed_everything(CONFIG['SEED'])
print(f"✓ Random seed set to {CONFIG['SEED']}")

✓ Random seed set to 42


In [4]:

print(f"\nLoading training data from: {CONFIG['TRAIN_PATH']}")
train_dataloader, val_dataloader, pipeline = load_data(
    train_path=CONFIG['TRAIN_PATH'],
    train_ratio=CONFIG['TRAIN_RATIO'],
    batchsize=CONFIG['BATCH_SIZE'],
    columns_to_drop=CONFIG['COLUMNS_TO_DROP'],
    sequence_length=CONFIG['SEQUENCE_LENGTH'],
    threshold=CONFIG['THRESHOLD'],
    threshold_value=CONFIG['THRESHOLD_VALUE'],
    threshold_column=CONFIG['THRESHOLD_COLUMN'],
    target=CONFIG['TARGET'],
    scaling=CONFIG['SCALING'],
    seed=CONFIG['SEED'],
    verbose=True
)

print(f"✓ Training batches: {len(train_dataloader)}")
print(f"✓ Validation batches: {len(val_dataloader)}")


Loading training data from: ../datasets_challenge/train
train_part_001.csv: Train=9473, Val=2368, Block=[3392:5760]
train_part_003.csv: Train=7895, Val=1973, Block=[7115:9088]
train_part_004.csv: Train=1145, Val=285, Block=[837:1122]
train_part_005.csv: Train=1221, Val=304, Block=[1159:1463]
train_part_006.csv: Train=16405, Val=4101, Block=[15872:19973]
train_part_007.csv: Train=1228, Val=306, Block=[426:732]
train_part_003.csv: Train=7895, Val=1973, Block=[7115:9088]
train_part_004.csv: Train=1145, Val=285, Block=[837:1122]
train_part_005.csv: Train=1221, Val=304, Block=[1159:1463]
train_part_006.csv: Train=16405, Val=4101, Block=[15872:19973]
train_part_007.csv: Train=1228, Val=306, Block=[426:732]
train_part_008.csv: Train=17777, Val=4443, Block=[14000:18443]
train_part_009.csv: Train=6176, Val=1543, Block=[2105:3648]
train_part_010.csv: Train=7567, Val=1891, Block=[7157:9048]
train_part_011.csv: Train=12416, Val=3103, Block=[7705:10808]
train_part_008.csv: Train=17777, Val=4443, B

In [7]:

all_train_targets = []
for i in range(len(train_dataloader.dataset)):
    _, y = train_dataloader.dataset[i]
    all_train_targets.append(y.item())

y_train_np = np.array(all_train_targets)

print(f"\n📊 TRAINING DATA:")
print(f"   Total samples: {len(y_train_np):,}")
print(f"   Mean: {y_train_np.mean():.4f}° | Std: {y_train_np.std():.4f}°")
print(f"   Range: [{y_train_np.min():.4f}°, {y_train_np.max():.4f}°]")

# Load CSV for bucket evaluation
dist_df = pd.read_csv(CONFIG['DIST_CSV'])
shot_band_mapping = dist_df.set_index("sideSlip")["band"].to_dict()

y_train_rounded = np.round(y_train_np, 1)
bands = pd.Series(y_train_rounded).map(shot_band_mapping).fillna("unknown")
band_counts = bands.value_counts()

print(f"\n📈 SHOT BAND DISTRIBUTION:")
for band in ['many', 'medium', 'few']:
    if band in band_counts.index:
        count = band_counts[band]
        pct = count / len(y_train_np) * 100
        print(f"   {band.capitalize():8s}: {count:7,} ({pct:5.2f}%)")

print(f"\n⚠️  Imbalance ratio (Many/Few): {band_counts.get('many', 0) / max(1, band_counts.get('few', 1)):.1f}x\n")


📊 TRAINING DATA:
   Total samples: 109,599
   Mean: 0.0306° | Std: 1.4638°
   Range: [-14.2100°, 11.6500°]

📈 SHOT BAND DISTRIBUTION:
   Many    :  99,538 (90.82%)
   Medium  :   7,613 ( 6.95%)
   Few     :   2,448 ( 2.23%)

⚠️  Imbalance ratio (Many/Few): 40.7x



In [11]:


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"DEVICE: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}\n")

scaler = GradScaler(device=device.type) if CONFIG['USE_AMP'] else None

X_sample, _ = next(iter(train_dataloader))
input_size = X_sample.shape[2]

# Create model
model = LSTMModel(
    input_size=input_size,
    output_size=1,
    rnn_units=CONFIG['RNN_UNITS'],
    fc_units=CONFIG['FC_UNITS'],
    dropout=CONFIG['DROPOUT'],
    num_layers=CONFIG['NUM_LAYERS'],
    use_batch_norm=True,
    bidirectional=CONFIG['BIDIRECTIONAL'],
    feature_dim=CONFIG['FEATURE_DIM']
).to(device)

# Create loss
criterion = EnhancedLDS_FocalR_Loss(
    train_targets=y_train_np,
    bucket_weights=CONFIG['BUCKET_WEIGHTS'],
    lds_kernel=CONFIG['LDS_KERNEL'],
    lds_ks=CONFIG['LDS_KS'],
    lds_sigma=CONFIG['LDS_SIGMA'],
    focal_gamma=CONFIG['FOCAL_GAMMA'],
    use_huber=CONFIG['USE_HUBER'],
    huber_beta=CONFIG['HUBER_BETA'],
    num_bins=CONFIG['NUM_BINS'],
    dist_csv_path=CONFIG['DIST_CSV'] 
).to(device)


total_params = sum(p.numel() for p in model.parameters())
print(f"\n======================================================================")
print(f"MODEL: {CONFIG['MODEL_TYPE']}")
print(f"======================================================================")
print(f"Parameters: {total_params:,}")
print(f"Size: {total_params * 4 / 1024**2:.2f} MB")
print(f"Architecture:")
print(f"  - RNN Units: {CONFIG['RNN_UNITS']}")
print(f"  - Layers: {CONFIG['NUM_LAYERS']}")
print(f"  - Bidirectional: {CONFIG['BIDIRECTIONAL']}")
print(f"  - Attention: {CONFIG['USE_ATTENTION']}")
print(f"======================================================================\n")

DEVICE: cpu

[EnhancedLDS+FocalR Loss - CSV-Aligned Configuration]
Loaded 234 band definitions from CSV
Approximate boundaries (max |sideslip| per band):
  Many-shot:   <= 2.4000 degrees
  Medium-shot: <= 5.0000 degrees
Data distribution:
  Many:    99538 (90.82%)
  Medium:   7613 ( 6.95%)
  Few:      2448 ( 2.23%)
Bucket weights: 1.0 / 3.0 / 8.0
Focal gamma: 1.0 | Huber beta: 0.08
LDS: kernel=gaussian, sigma=2.0


MODEL: BiLSTM_Attention
Parameters: 675,885
Size: 2.58 MB
Architecture:
  - RNN Units: 128
  - Layers: 2
  - Bidirectional: True
  - Attention: True


[EnhancedLDS+FocalR Loss - CSV-Aligned Configuration]
Loaded 234 band definitions from CSV
Approximate boundaries (max |sideslip| per band):
  Many-shot:   <= 2.4000 degrees
  Medium-shot: <= 5.0000 degrees
Data distribution:
  Many:    99538 (90.82%)
  Medium:   7613 ( 6.95%)
  Few:      2448 ( 2.23%)
Bucket weights: 1.0 / 3.0 / 8.0
Focal gamma: 1.0 | Huber beta: 0.08
LDS: kernel=gaussian, sigma=2.0


MODEL: BiLSTM_Attention


In [ ]:
# filepath: notebooks/task2v3.ipynb
# ============================================================================
# TRAINING SETUP
# ============================================================================

# Augmentation
mixup = MixupAugmentation(
    alpha=CONFIG['MIXUP_ALPHA'],
    prob=CONFIG['MIXUP_PROB'],
    prioritize_rare=CONFIG['MIXUP_PRIORITIZE_RARE'],
    rare_threshold=CONFIG['MIXUP_RARE_THRESHOLD']
)

temporal_aug = TemporalAugmentation(
    noise_prob=CONFIG['TEMPORAL_NOISE_PROB'],
    noise_std=CONFIG['TEMPORAL_NOISE_STD'],
    mask_prob=CONFIG['TEMPORAL_MASK_PROB'],
    mask_ratio=CONFIG['TEMPORAL_MASK_RATIO'],
    scale_prob=CONFIG['TEMPORAL_SCALE_PROB'],
    scale_range=CONFIG['TEMPORAL_SCALE_RANGE']
)

# Optimizer & Scheduler
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CONFIG['LEARNING_RATE'],
    weight_decay=CONFIG['WEIGHT_DECAY'],
    betas=CONFIG['BETAS']
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=CONFIG['SCHEDULER_T_MAX'],
    eta_min=CONFIG['SCHEDULER_ETA_MIN']
)

print(f"✓ Optimizer: AdamW (lr={CONFIG['LEARNING_RATE']:.1e})")
print(f"✓ Augmentation: Mixup {CONFIG['MIXUP_PROB']:.0%}, Temporal {CONFIG['TEMPORAL_NOISE_PROB']:.0%}")

# Training state
train_curve, val_curve = [], []
train_mae_curve, val_mae_curve = [], []
bucket_many, bucket_medium, bucket_few = [], [], []
learning_rate_curve = []

best_few_shot_mae = np.inf
best_epoch = -1

print(f"\n🎯 CHECKPOINT STRATEGY:")
print(f"   Save model with LOWEST few-shot MAE")
print(f"   Constraint: Many-shot MAE ≤ {CONFIG['MANY_SHOT_CONSTRAINT']:.2f}°\n")

In [ ]:
# filepath: notebooks/task2v3.ipynb
# ============================================================================
# STREAMLINED TRAINING LOOP
# ============================================================================

for epoch in range(CONFIG['NUM_EPOCHS']):
    # =============== TRAINING ===============
    model.train()
    running_loss, running_mae = 0.0, 0.0

    for step, (xb, yb) in enumerate(train_dataloader):
        xb, yb = xb.to(device), yb.to(device).reshape(-1, 1)
        
        # Apply augmentation
        if epoch < CONFIG['AUGMENTATION_EPOCHS']:
            xb = temporal_aug(xb)
        xb, yb = mixup(xb, yb)

        optimizer.zero_grad(set_to_none=True)
        
        # Forward pass with AMP
        if CONFIG['USE_AMP'] and device.type == "cuda":
            with autocast(device_type='cuda'):
                pred = model(xb)
                total_loss = criterion(pred, yb)
        else:
            pred = model(xb)
            total_loss = criterion(pred, yb)
        
        # Backward pass
        if scaler is not None:
            scaler.scale(total_loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['GRADIENT_CLIP'])
            scaler.step(optimizer)
            scaler.update()
        else:
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['GRADIENT_CLIP'])
            optimizer.step()

        running_loss += total_loss.item()
        running_mae += (pred - yb).abs().mean().item()

    avg_train_loss = running_loss / len(train_dataloader)
    avg_train_mae = running_mae / len(train_dataloader)
    train_curve.append(avg_train_loss)
    train_mae_curve.append(avg_train_mae)

    # =============== VALIDATION ===============
    model.eval()
    running_val_loss, running_val_mae = 0.0, 0.0
    all_val_preds, all_val_targets = [], []

    with torch.no_grad():
        for xv, yv in val_dataloader:
            xv, yv = xv.to(device), yv.to(device).reshape(-1, 1)
            pred_val = model(xv)
            total_loss_val = criterion(pred_val, yv)
            
            running_val_loss += total_loss_val.item()
            running_val_mae += (pred_val - yv).abs().mean().item()
            all_val_preds.append(pred_val)
            all_val_targets.append(yv)

    all_val_preds = torch.cat(all_val_preds, dim=0)
    all_val_targets = torch.cat(all_val_targets, dim=0)
    bucket_stats = criterion.bucket_metrics(all_val_preds, all_val_targets)

    avg_val_loss = running_val_loss / len(val_dataloader)
    avg_val_mae = running_val_mae / len(val_dataloader)
    val_curve.append(avg_val_loss)
    val_mae_curve.append(avg_val_mae)
    
    bucket_many.append(bucket_stats['mae_many'])
    bucket_medium.append(bucket_stats['mae_medium'])
    bucket_few.append(bucket_stats['mae_few'])

    current_lr = optimizer.param_groups[0]['lr']
    learning_rate_curve.append(current_lr)
    scheduler.step()

    # =============== LOGGING ===============
    many_mae = bucket_stats['mae_many']
    few_mae = bucket_stats['mae_few']
    
    print(f"Epoch {epoch+1:3d}/{CONFIG['NUM_EPOCHS']} | "
          f"Loss: {avg_val_loss:.5f} | MAE: {avg_val_mae:.5f} | "
          f"Many: {many_mae:.4f} | "
          f"Med: {bucket_stats['mae_medium']:.4f} | "
          f"Few: {few_mae:.4f}", end="")

    # =============== CHECKPOINTING ===============
    save_checkpoint = False
    checkpoint_reason = ""
    
    if few_mae < best_few_shot_mae:
        if many_mae <= CONFIG['MANY_SHOT_CONSTRAINT']:
            save_checkpoint = True
            checkpoint_reason = f"✅ BEST (Few↓ {best_few_shot_mae:.4f}→{few_mae:.4f}, Many={many_mae:.4f})"
            best_few_shot_mae = few_mae
            best_epoch = epoch
        else:
            save_checkpoint = True
            checkpoint_reason = f"⚠️ SAVE (Few↓ but Many={many_mae:.4f} > 0.2°)"
            best_few_shot_mae = few_mae
            best_epoch = epoch
    
    if save_checkpoint:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'val_loss': avg_val_loss,
            'val_mae': avg_val_mae,
            'bucket_stats': bucket_stats,
            'many_mae': many_mae,
            'few_mae': few_mae,
            'config': CONFIG
        }, CONFIG['CHECKPOINT_NAME'])
        
        print(f" ← {checkpoint_reason}")
    elif epoch - best_epoch >= CONFIG['EARLY_STOP_PATIENCE']:
        print(f"\n\n⏹ Early stopping at epoch {epoch+1}")
        print(f"   Best Few-shot MAE: {best_few_shot_mae:.4f} (Epoch {best_epoch+1})")
        break
    else:
        print("")

print(f"\n{'='*80}")
print(f"✓ TRAINING COMPLETE")
print(f"  Best Epoch: {best_epoch+1}")
print(f"  Best Few-shot MAE: {best_few_shot_mae:.5f}")
print(f"{'='*80}\n")

In [ ]:


checkpoint = torch.load(CONFIG['CHECKPOINT_NAME'], map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.to(device).eval()

print(f"✓ Loaded checkpoint: {CONFIG['CHECKPOINT_NAME']}")
print(f"  Epoch: {checkpoint['epoch']+1}")
print(f"  Val Loss: {checkpoint['val_loss']:.5f}")
print(f"  Val MAE: {checkpoint['val_mae']:.5f}")

print(f"  Many:   {checkpoint['bucket_stats']['mae_many']:.4f}°")
print(f"  Medium: {checkpoint['bucket_stats']['mae_medium']:.4f}°")
print(f"  Few:    {checkpoint['bucket_stats']['mae_few']:.4f}°")

In [ ]:


fig, axes = plt.subplots(2, 2, figsize=CONFIG['FIG_SIZE'])

# Loss curves
axes[0, 0].plot(train_curve, label="Train", color='blue', linewidth=2)
axes[0, 0].plot(val_curve, label="Val", color='orange', linewidth=2)
axes[0, 0].axvline(best_epoch, color='red', linestyle='--', label='Best')
axes[0, 0].set_xlabel("Epoch")
axes[0, 0].set_ylabel("Loss")
axes[0, 0].set_title("Training & Validation Loss")
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# MAE curves
axes[0, 1].plot(train_mae_curve, label="Train", color='green', linewidth=2)
axes[0, 1].plot(val_mae_curve, label="Val", color='red', linewidth=2)
axes[0, 1].axvline(best_epoch, color='red', linestyle='--', label='Best')
axes[0, 1].set_xlabel("Epoch")
axes[0, 1].set_ylabel("MAE (°)")
axes[0, 1].set_title("Mean Absolute Error")
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# Bucket MAE evolution
axes[1, 0].plot(bucket_many, label="Many-shot", color='blue', linewidth=2)
axes[1, 0].plot(bucket_medium, label="Medium-shot", color='orange', linewidth=2)
axes[1, 0].plot(bucket_few, label="Few-shot", color='red', linewidth=2)
axes[1, 0].axvline(best_epoch, color='red', linestyle='--')
axes[1, 0].axhline(0.2, color='black', linestyle=':', label='Constraint')
axes[1, 0].set_xlabel("Epoch")
axes[1, 0].set_ylabel("Bucket MAE (°)")
axes[1, 0].set_title("Bucket-wise Performance")
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

# Generalization gap
axes[1, 1].plot(train_curve, label="Train", color='blue', linewidth=2)
axes[1, 1].plot(val_curve, label="Val", color='orange', linewidth=2)
axes[1, 1].fill_between(range(len(train_curve)), train_curve, val_curve, alpha=0.2, color='gray')
axes[1, 1].axvline(best_epoch, color='red', linestyle='--')
axes[1, 1].set_xlabel("Epoch")
axes[1, 1].set_ylabel("Loss")
axes[1, 1].set_title("Generalization Gap")
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n📈 Training Summary:")
print(f"   Best Epoch: {best_epoch+1}/{len(train_curve)}")
print(f"   Best Val Loss: {min(val_curve):.5f}")
print(f"   Final Bucket MAE: Many={bucket_many[best_epoch]:.4f}° | "
      f"Med={bucket_medium[best_epoch]:.4f}° | Few={bucket_few[best_epoch]:.4f}°")

In [ ]:


def predict_with_tta(model, dataloader, device, config):
    """Lightweight TTA: 3 augmentations with uniform weighting."""
    model.eval()
    all_preds = []
    
    n_aug = config['TTA_N_AUGMENTATIONS']
    noise_levels = config['TTA_NOISE_LEVELS']
    
    with torch.no_grad():
        for x, y in dataloader:
            x, y = x.to(device), y.to(device).view(-1, 1)
            pred = model(x)
            batch_preds = [pred.cpu() * config['TTA_ORIGINAL_WEIGHT']]
            
            for noise_std in noise_levels:
                x_aug = x + torch.randn_like(x) * noise_std
                pred_aug = model(x_aug)
                batch_preds.append(pred_aug.cpu())
            
            weights = torch.tensor([config['TTA_ORIGINAL_WEIGHT']] + [1.0] * len(noise_levels))
            weights = weights / weights.sum()
            avg_pred = torch.stack(batch_preds).mul(weights.view(-1, 1, 1)).sum(dim=0)
            
            all_preds.append(avg_pred.squeeze(-1).numpy())
    
    return np.concatenate(all_preds)

print("✓ Simplified TTA function defined (3x faster)")

In [ ]:


print(f"\nLoading test data from: {CONFIG['TEST_PATH']}")
test_dataloader = load_test_data(
    test_path=CONFIG['TEST_PATH'],
    batchsize=CONFIG['BATCH_SIZE'],
    pipeline=pipeline,
    load_anonymized_data=True,
    verbose=False
)

print(f"Generating predictions with TTA (n_aug={CONFIG['TTA_N_AUGMENTATIONS']})...")
test_preds = predict_with_tta(model, test_dataloader, device, CONFIG)

print(f"\n✓ Generated {len(test_preds)} predictions")
print(f"  Range: [{test_preds.min():.2f}°, {test_preds.max():.2f}°]")
print(f"  Mean: {test_preds.mean():.3f}° | Std: {test_preds.std():.3f}°")

# Sanity checks
assert len(test_preds) > 0, "No predictions generated!"
assert not np.isnan(test_preds).any(), "NaN values in predictions!"
assert not np.isinf(test_preds).any(), "Inf values in predictions!"
print("✓ Predictions passed sanity checks")

In [ ]:


test_ground_truths = []
with torch.no_grad():
    for x, y in test_dataloader:
        test_ground_truths.append(y.numpy())
test_ground_truths = np.concatenate(test_ground_truths)

fig = go.Figure()
fig.add_trace(go.Scatter(y=test_ground_truths, mode='lines', name='Placeholder (zeros)', 
                         line=dict(color='blue', width=1)))
fig.add_trace(go.Scatter(y=test_preds.flatten(), mode='lines', name='Predictions', 
                         line=dict(color='red', width=1), opacity=0.7))
fig.update_layout(
    title='Test Set Predictions',
    xaxis_title='Sample Index',
    yaxis_title='Sideslip Angle (°)',
    width=CONFIG['PLOTLY_WIDTH'],
    height=CONFIG['PLOTLY_HEIGHT']
)
fig.show()

In [ ]:


y_pred = pd.DataFrame({
    "id": range(len(test_preds)),
    "predicted_sideslip": test_preds
})

y_pred.to_csv(CONFIG['SUBMISSION_CSV'], index=False)

print(f"✅ Predictions saved to: {CONFIG['SUBMISSION_CSV']}")
print(f"   Total samples: {len(test_preds)}")
print(f"   Ready for submission!")